# Pipeline Medallion — `ecommerce_itens_pedido`

Este notebook implementa a arquitetura **Medallion (Bronze → Silver → Gold)** para a tabela de itens de pedido do e-commerce, utilizando **PySpark no Databricks** com armazenamento no **Azure Data Lake Storage Gen2 (ADLS)** e escrita no formato **Delta Lake**.

---

## 🗂️ Fluxo do Pipeline

```
RAW (CSV no ADLS)
      │
      ▼
  🥉 BRONZE  — Ingestão bruta com metadados de auditoria
      │
      ▼
  🥈 SILVER  — Limpeza, enriquecimento e aplicação de regras de negócio
      │
      ▼
  🥇 GOLD    — Data Marts agregados (KPIs) para consumo analítico
      │
      ▼
  🗄️ SQL SERVER — Exportação dos KPIs para consumo em relatórios/BI
```

---

## 📐 Regras de Negócio Aplicadas (Silver)

| # | Regra | Descrição |
|---|-------|-----------|
| 1 | **Deduplicação e nulidade** | Remoção de registros sem `id_item_pedido` e eliminação de duplicatas pela mesma chave |
| 2 | **Tipagem condicional de quantidade** | `INTEGER` para itens vendidos por unidade (`un`, `pc`, `cx`); `DOUBLE` para itens vendidos por peso/volume (`kg`, `L`) — lookup via join com `ecommerce_produtos` |
| 3 | **Normalização de valores monetários** | Substituição de vírgula por ponto em `preco_unitario` e `desconto_aplicado`, seguida de cast para `Decimal(10,2)` |
| 4 | **Coluna derivada: `valor_liquido_item`** | Calculado como `(preco_unitario - desconto_aplicado) × quantidade`, representando o valor efetivo de cada linha do pedido |
| 5 | **Particionamento histórico** | A partição `ano_particao` / `mes_particao` é derivada da **data do pedido** (`dt_pedido`), e não da data de ingestão — garantindo a correta organização temporal dos dados |

---

## 📊 KPIs Gerados (Gold)

| Tabela Gold | Descrição | Granularidade |
|-------------|-----------|---------------|
| `gold_kpi_receita_desconto_categoria` | Receita líquida total e taxa média de desconto por categoria raiz | Ano / Mês / Categoria |
| `gold_kpi_sku_trimestre` | Receita total e quantidade vendida por SKU | Ano / Trimestre / SKU |
| `gold_kpi_complexidade_carrinho` | Média de itens por pedido (ticket de complexidade) | Ano / Mês |
| `gold_kpi_yoy_subcategoria` | Receita por subcategoria com crescimento YoY (%) calculado via Window Function | Ano / Subcategoria |

---

## 🔧 Dependências e Pré-requisitos

- **Variáveis de ambiente** (arquivo `../env`): `CLIENT_ID`, `TENANT_ID`, `CLIENT_SECRET`, `STORAGE_ACCOUNT_NAME`, `SQL_HOST`, `SQL_DATABASE`, `SQL_USERNAME`, `SQL_PASSWORD`
- **Tabelas Silver auxiliares já processadas**: `ecommerce_produtos` (Delta) e `ecommerce_pedidos` (Delta)
- **Tabela Silver auxiliar para hierarquia de categorias**: `ecommerce_categorias` (Delta)
- **Autenticação no ADLS**: OAuth 2.0 via Service Principal (Client Credentials)

## 1. Setup e Credenciais

Carrega as variáveis de ambiente a partir do arquivo `../env` e configura:
- As opções de autenticação OAuth 2.0 para acesso ao ADLS Gen2 via Service Principal
- Os caminhos base das camadas **Raw** e **Bronze** no Data Lake

> ⚠️ Nenhuma credencial é hardcoded. Todas as chaves sensíveis são lidas exclusivamente via `os.getenv()`.

In [0]:
# ============================================================
# 1. SETUP E CREDENCIAIS (ecommerce_itens_pedido)
# ============================================================

import os
from datetime import date

from dotenv import load_dotenv

from pyspark.sql.functions import (
    current_timestamp,
    year,
    month,
    col,
    sum,
    round,
    date_format,
    when,
    regexp_replace,
    count,
    avg,
    ceil,
    lag,
    lit
)

from pyspark.sql.window import Window


# ============================================================
# 1.1 CARREGAMENTO DO .ENV + FALLBACK PARA JOB PARAMETERS
# ============================================================
# Objetivo:
# - Continuar funcionando manualmente com .env
# - Funcionar via Databricks Job usando Job Parameters
# - Evitar spark.conf.set(), pois no Databricks Free/Serverless
#   algumas configs fs.azure.* podem não estar disponíveis
# ============================================================

try:
    load_dotenv("../env")
    load_dotenv("../.env")
    load_dotenv(".env")
    print("Tentativa de carregamento do .env realizada.")
except Exception as e:
    print(f"Não foi possível carregar .env. Seguindo com fallback. Detalhe: {e}")


def get_config_value(name: str, required: bool = True, default: str = "") -> str:
    """
    Busca uma configuração na seguinte ordem:

    1. Variáveis de ambiente carregadas pelo .env
    2. Databricks Job Parameters, via dbutils.widgets.get()
    3. Valor default, quando informado

    Isso permite que o notebook funcione tanto em execução manual
    quanto em execução agendada pelo Databricks Job.
    """

    value = os.getenv(name)

    if value is None or str(value).strip() == "":
        try:
            value = dbutils.widgets.get(name)
        except Exception:
            value = None

    if value is None or str(value).strip() == "":
        value = default

    value = str(value).strip() if value is not None else ""

    if required and value == "":
        raise ValueError(
            f"Configuração obrigatória não encontrada: {name}. "
            f"Verifique se ela existe no .env ou nos Job Parameters do Databricks."
        )

    return value


# ============================================================
# 1.2 VARIÁVEIS ADLS GEN2
# ============================================================

client_id = get_config_value("CLIENT_ID")
tenant_id = get_config_value("TENANT_ID")
client_secret = get_config_value("CLIENT_SECRET")
storage_account = get_config_value("STORAGE_ACCOUNT_NAME")

# Pelo seu path original, o arquivo está em:
# abfss://raw@storage/batch-data/ecommerce_itens_pedido.csv
source_folder = get_config_value("CONTAINER_NAME", required=False, default="batch-data")

# Containers usados no projeto
raw_container = get_config_value("RAW_CONTAINER", required=False, default="raw")
squad_container = get_config_value("SQUAD_CONTAINER", required=False, default="squad3")


# ============================================================
# 1.3 OPÇÕES DE AUTENTICAÇÃO ADLS GEN2
# ============================================================
# Importante:
# NÃO usar spark.conf.set() aqui.
#
# A estratégia é manter as opções em dicionário e usar nas leituras
# e escritas com .options(**adls_options).
# ============================================================

storage_account_fqdn = f"{storage_account}.dfs.core.windows.net"

adls_options = {
    f"fs.azure.account.auth.type.{storage_account_fqdn}": "OAuth",
    f"fs.azure.account.oauth.provider.type.{storage_account_fqdn}": "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
    f"fs.azure.account.oauth2.client.id.{storage_account_fqdn}": client_id,
    f"fs.azure.account.oauth2.client.secret.{storage_account_fqdn}": client_secret,
    f"fs.azure.account.oauth2.client.endpoint.{storage_account_fqdn}": f"https://login.microsoftonline.com/{tenant_id}/oauth2/token"
}

# Mantido também em formato genérico, caso alguma célula posterior
# esteja usando esse padrão antigo de opções.
adls_options_generic = {
    "fs.azure.account.auth.type": "OAuth",
    "fs.azure.account.oauth.provider.type": "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
    "fs.azure.account.oauth2.client.id": client_id,
    "fs.azure.account.oauth2.client.secret": client_secret,
    "fs.azure.account.oauth2.client.endpoint": f"https://login.microsoftonline.com/{tenant_id}/oauth2/token"
}


# ============================================================
# 1.4 PATHS DA TABELA
# ============================================================

tabela = "ecommerce_itens_pedido"

path_raw = (
    f"abfss://{raw_container}@{storage_account}.dfs.core.windows.net/"
    f"{source_folder}/{tabela}.csv"
)

path_bronze = (
    f"abfss://{squad_container}@{storage_account}.dfs.core.windows.net/"
    f"bronze/{tabela}"
)


# ============================================================
# 1.5 PERÍODO DE REFERÊNCIA DA EXECUÇÃO
# ============================================================

_hoje = date.today()
ANO_EXEC = _hoje.year
MES_EXEC = _hoje.month
DIA_EXEC = _hoje.day


# ============================================================
# 1.6 LOG SEGURO DE VALIDAÇÃO
# ============================================================

print(f"Configuração finalizada para a tabela: {tabela}")
print(f"Período de execução: {DIA_EXEC}/{MES_EXEC}/{ANO_EXEC}")
print(f"Storage Account: {storage_account}")
print(f"Raw Container: {raw_container}")
print(f"Source Folder: {source_folder}")
print(f"Squad Container: {squad_container}")
print(f"Path Raw: {path_raw}")
print(f"Path Bronze: {path_bronze}")
print("Credenciais carregadas sem expor secrets.")

## 2. 🥉 Camada Bronze — Ingestão RAW → Bronze

Realiza a ingestão do arquivo CSV bruto da camada Raw para a camada Bronze, **sem transformações de negócio** — apenas acréscimo de metadados de rastreabilidade:

| Coluna adicionada | Descrição |
|-------------------|-----------|
| `bronze_ingested_at` | Timestamp do momento da ingestão |
| `bronze_source_file` | Caminho completo do arquivo de origem (via `_metadata.file_path`) |
| `ano_particao` | Ano extraído do timestamp de ingestão — usado para particionamento físico |
| `mes_particao` | Mês extraído do timestamp de ingestão — usado para particionamento físico |

> 📌 O schema é lido como `string` (`inferSchema=false`) para preservar os dados exatamente como estão na fonte, evitando casting automático incorreto. A tipagem adequada ocorre na camada Silver.
>
> 📌 O modo de escrita é `append`, permitindo múltiplas execuções incrementais sem sobrescrever cargas anteriores.

In [0]:
# ============================================================
# 2. EXTRAÇÃO E CARGA (RAW -> BRONZE)
# ============================================================
print(f"Lendo {tabela} da camada Raw...")

# Leitura do CSV sem inferência de schema para preservar os dados originais
df_raw = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "false")  # Schema todo como string; tipagem será feita na Silver
    .options(**adls_options) 
    .load(path_raw)
)

timestamp_carga = current_timestamp()

# Enriquecimento com colunas de auditoria e particionamento por data de ingestão
df_bronze = (
    df_raw
    .withColumn("bronze_ingested_at", timestamp_carga)
    .withColumn("bronze_source_file", col("_metadata.file_path"))  # Caminho do arquivo de origem
    .withColumn("ano_particao", year(timestamp_carga))
    .withColumn("mes_particao", month(timestamp_carga))
)

print(f"Gravando fisicamente na camada Bronze: {path_bronze}")

(
    df_bronze.write
    .format("delta")
    .mode("append")  # Append garante idempotência em reprocessamentos incrementais
    .options(**adls_options)
    .partitionBy("ano_particao", "mes_particao")
    .save(path_bronze)
)

print("Ingestão Bronze finalizada com sucesso!")

## 3. 🥈 Camada Silver — Limpeza, Enriquecimento e Regras de Negócio

Esta é a etapa central de transformação do pipeline. A Silver recebe os dados brutos da Bronze e aplica todas as regras de qualidade e negócio antes de disponibilizar os dados para agregação.

### Fontes de dados utilizadas

| DataFrame | Camada | Formato | Uso |
|-----------|--------|---------|-----|
| `df_bronze_itens` | Bronze | Delta | Dados brutos da tabela principal |
| `df_silver_produtos` | Silver | Delta | Lookup de `unidade_medida` por SKU (Regra 2) |
| `df_silver_pedidos` | Silver | Delta | Lookup de `dt_pedido` por `id_pedido` (Regra 5) |

### Etapas de transformação

1. **Limpeza básica (Regra 1):** Remove linhas sem `id_item_pedido` e elimina duplicatas pela mesma chave
2. **Joins de enriquecimento:** Left join com `ecommerce_produtos` para obter `unidade_medida`; left join com `ecommerce_pedidos` para obter `dt_pedido`
3. **Tipagem e normalização (Regras 2 e 3):** Cast condicional de `quantidade` e normalização decimal de valores monetários
4. **Coluna derivada (Regra 4):** Cálculo de `valor_liquido_item = (preco_unitario - desconto_aplicado) × quantidade`
5. **Particionamento histórico (Regra 5):** `ano_particao` e `mes_particao` derivados de `dt_pedido`, não da data de processamento
6. **Seleção final:** Remoção das colunas auxiliares dos joins antes da escrita

### Schema de saída da Silver

| Coluna | Tipo | Descrição |
|--------|------|-----------|
| `id_item_pedido` | String | Chave primária do item |
| `id_pedido` | String | FK para a tabela de pedidos |
| `sku` | String | Código do produto |
| `quantidade` | Integer / Double | Quantidade (tipo depende da `unidade_medida`) |
| `preco_unitario` | Decimal(10,2) | Preço unitário normalizado |
| `desconto_aplicado` | Decimal(10,2) | Desconto por unidade |
| `valor_liquido_item` | Decimal(10,2) | Valor efetivo da linha do pedido |
| `ano_particao` | Integer | Ano do pedido (partição física) |
| `mes_particao` | Integer | Mês do pedido (partição física) |
| `silver_processed_at` | Timestamp | Timestamp de processamento para auditoria |

In [0]:
# ============================================================
# 3. CAMADA SILVER (ENRIQUECIMENTO E REGRAS COMPLEXAS)
# ============================================================

print(f"Iniciando processamento avançado da camada Silver para: {tabela}...")

# Caminhos das tabelas
path_silver_itens = f"abfss://squad3@{storage_account}.dfs.core.windows.net/silver/{tabela}"
path_silver_produtos = f"abfss://squad3@{storage_account}.dfs.core.windows.net/silver/ecommerce_produtos"
path_silver_pedidos = f"abfss://squad3@{storage_account}.dfs.core.windows.net/silver/ecommerce_pedidos"

# 1. LEITURA (Bronze atual + Tabelas Silver auxiliares)
# Todas as tabelas auxiliares estão em Delta
df_bronze_itens = spark.read.format("delta").options(**adls_options).load(path_bronze)
df_silver_produtos = spark.read.format("delta").options(**adls_options).load(path_silver_produtos)
df_silver_pedidos = spark.read.format("delta").options(**adls_options).load(path_silver_pedidos)

# 2. LIMPEZA BÁSICA (Regra 1)
# Remove nulos na chave primária e duplicatas antes de qualquer transformação
df_itens_clean = (
    df_bronze_itens
    .dropna(subset=["id_item_pedido"])
    .dropDuplicates(["id_item_pedido"])
    .withColumn("id_item_pedido", col("id_item_pedido").cast("string"))
    .withColumn("id_pedido", col("id_pedido").cast("string"))
    .withColumn("sku", col("sku").cast("string"))
)

# # 3. JOINS PARA ENRIQUECIMENTO (Buscando dados necessários para as regras)
# # Pegando apenas a unidade de medida dos produtos
# df_prod_aux = df_silver_produtos.select("sku", col("unidade_medida").alias("aux_unidade_medida"))
# df_itens_enriquecido = df_itens_clean.join(df_prod_aux, on="sku", how="left")

# # Pegando a data do pedido (AJUSTE O NOME DA COLUNA 'dt_pedido' SE NECESSÁRIO)
# df_ped_aux = df_silver_pedidos.select("id_pedido", col("dt_pedido").alias("aux_data_pedido"))
# df_itens_enriquecido = df_itens_enriquecido.join(df_ped_aux, on="id_pedido", how="left")

# 3. JOINS PARA ENRIQUECIMENTO (Corrigindo a Explosão de Cardinalidade)

# Deduplica a tabela de produtos pelo SKU antes do Join
df_prod_aux = (
    df_silver_produtos
    .select("sku", col("unidade_medida").alias("aux_unidade_medida"))
    .dropDuplicates(["sku"])
)
df_itens_enriquecido = df_itens_clean.join(df_prod_aux, on="sku", how="left")

# Deduplica a tabela de pedidos pelo ID do pedido antes do Join
df_ped_aux = (
    df_silver_pedidos
    .select("id_pedido", col("dt_pedido").alias("aux_data_pedido"))
    .dropDuplicates(["id_pedido"])
)
df_itens_enriquecido = df_itens_enriquecido.join(df_ped_aux, on="id_pedido", how="left")



# 4. APLICAÇÃO DAS REGRAS FINANCEIRAS E TIPAGEM (Regras 2, 3, 4 e 5)
df_silver = (
    df_itens_enriquecido
    
    # Regra 3: Tratamento e cast de preços para Decimal(10,2)
    # regexp_replace trata fontes com separador decimal em vírgula (padrão pt-BR)
    .withColumn("preco_unitario", regexp_replace(col("preco_unitario"), ",", ".").cast("decimal(10,2)"))
    .withColumn("desconto_aplicado", regexp_replace(col("desconto_aplicado"), ",", ".").cast("decimal(10,2)"))
    
    # Regra 2: Quantidade condicional (INT para unidades/peças, FLOAT para kg/L)
    .withColumn("quantidade", 
        when(col("aux_unidade_medida").isin("un", "pc", "cx"), regexp_replace(col("quantidade"), ",", ".").cast("integer"))
        .otherwise(regexp_replace(col("quantidade"), ",", ".").cast("double"))
    )
    
    # Regra 4: Coluna derivada (Valor Líquido do Item)
    # valor_liquido_item = (preco_unitario - desconto_aplicado) * quantidade
    .withColumn("valor_liquido_item", 
        ((col("preco_unitario") - col("desconto_aplicado")) * col("quantidade")).cast("decimal(10,2)")
    )
    
    # Regra 5: Particionamento baseado na data do pedido (e não na data atual)
    # Garante que os dados sejam alocados na partição histórica correta
    .withColumn("ano_particao", year("aux_data_pedido"))
    .withColumn("mes_particao", month("aux_data_pedido"))
    
    # Auditoria
    .withColumn("silver_processed_at", current_timestamp())
)

# 5. SELEÇÃO FINAL (Removendo as colunas auxiliares que trouxemos dos Joins)
df_silver_final = df_silver.select(
    "id_item_pedido", "id_pedido", "sku", "quantidade", 
    "preco_unitario", "desconto_aplicado", "valor_liquido_item", 
    "ano_particao", "mes_particao", "silver_processed_at"
)


# 6. GRAVAÇÃO NA SILVER
print(f"Gravando dados refinados em: {path_silver_itens}")

(
    df_silver_final.write
    .format("delta")
    .mode("append")  
    .options(**adls_options)
    .partitionBy("ano_particao", "mes_particao")
    .save(path_silver_itens)
)

print("SUCESSO! Tabela de Itens salva na Silver com Joins, cálculos financeiros e particionamento histórico.\n")
display(df_silver_final.limit(5))

## 4. 🥇 Camada Gold — Data Marts Agregados (KPIs)

A camada Gold consome o `df_silver_final` e tabelas de dimensão para construir **4 Data Marts analíticos** orientados a KPIs de negócio. Estes Data Marts seguem princípios de **Star Schema**, com a Silver de itens como tabela fato e produtos/categorias como dimensões.

### Hierarquia de Categorias

A tabela `ecommerce_categorias` é auto-relacionada (self-join) para resolver a hierarquia subcategoria → categoria raiz, expondo as colunas `nome_subcategoria` e `nome_categoria_raiz` no `df_base`.

### Colunas derivadas no `df_base`

| Coluna | Cálculo | Finalidade |
|--------|---------|------------|
| `trimestre` | `CEIL(mes_particao / 3)` | Agrupamento trimestral |
| `valor_bruto_item` | `preco_unitario × quantidade` | Base para cálculo de taxa de desconto |
| `valor_desconto_total` | `desconto_aplicado × quantidade` | Valor absoluto de desconto por item |

### KPIs Gerados

#### `gold_kpi_receita_desconto_categoria`
Receita líquida total e taxa de desconto média por categoria raiz, mês a mês.  
- `taxa_desconto_media = SUM(valor_desconto_total) / SUM(valor_bruto_item)`

#### `gold_kpi_sku_trimestre`
**Top 10 SKUs por trimestre**, por receita e por quantidade vendida. Cada SKU recebe um `rank_receita` e um `rank_quantidade` (via `row_number()` particionado por `ano_particao, trimestre`), e a tabela final mantém a **união** dos dois rankings — um SKU pode estar no top 10 de receita sem estar no de quantidade, e vice-versa. Resultado: até 20 SKUs por trimestre, cada um com ambos os ranks visíveis para o time filtrar pelo critério desejado.

#### `gold_kpi_complexidade_carrinho`
Média de **itens (linhas distintas) por pedido** — `COUNT(id_item_pedido)` agrupado por pedido, depois `AVG` por mês. Mede a complexidade/diversidade do carrinho (quantos produtos diferentes), e não o volume comprado — por isso não soma a coluna `quantidade`, que mistura unidades e kg/L conforme a Regra 2 da Silver e inflaria pedidos de produtos vendidos a peso/volume.

#### `gold_kpi_yoy_subcategoria`
Crescimento Year-over-Year (YoY) da receita por subcategoria, calculado via **Window Function** com `lag()`, particionada por `id_categoria` — a chave real da subcategoria, e não o nome. Isso evita que duas subcategorias de raízes diferentes com o mesmo nome (ex.: "Acessórios" em Eletrônicos e em Moda) tenham sua receita somada/comparada como se fossem uma única série. `nome_subcategoria` continua na tabela apenas para exibição. A variação percentual é dada por:  
`crescimento_yoy_perc = ((receita_ano_atual - receita_ano_anterior) / receita_ano_anterior) × 100`

> 📌 A escrita Gold utiliza **delete + append**, com as colunas que identificam o lote (`delete_cols`) definidas individualmente por tabela — por exemplo, `gold_kpi_sku_trimestre` usa `trimestre` em vez de `mes_particao`, e `gold_kpi_yoy_subcategoria` usa só `ano_particao` + `dia_exec`. Isso evita o erro de coluna inexistente que antes fazia o delete falhar silenciosamente nessas duas tabelas (mascarado por um `try/except` genérico), acumulando duplicatas a cada execução semanal.

In [0]:
# ============================================================
# 4. CAMADA GOLD (DATA MARTS AGREGADOS - STAR SCHEMA)
# ============================================================

from pyspark.sql.functions import row_number

print("Iniciando construção dos Data Marts agregados na camada Gold...")

# ------------------------------------------------------------
# 1. LEITURA DAS TABELAS SILVER E JOINS BASE
# ------------------------------------------------------------
path_silver_produtos   = f"abfss://squad3@{storage_account}.dfs.core.windows.net/silver/ecommerce_produtos"
path_silver_categorias = f"abfss://squad3@{storage_account}.dfs.core.windows.net/silver/ecommerce_categorias"

df_dim_produtos    = spark.read.format("delta").options(**adls_options).load(path_silver_produtos)
df_dim_categorias  = spark.read.format("delta").options(**adls_options).load(path_silver_categorias)

# Self-join em ecommerce_categorias para resolver a hierarquia subcategoria -> categoria raiz
df_categorias_hierarquia = (
    df_dim_categorias.alias("sub")
    .join(
        df_dim_categorias.alias("raiz"),
        col("sub.id_categoria_pai") == col("raiz.id_categoria"),
        "left"
    )
    .select(
        col("sub.id_categoria").alias("id_categoria"),
        col("sub.nome_categoria").alias("nome_subcategoria"),
        col("raiz.nome_categoria").alias("nome_categoria_raiz")
    )
)

# DataFrame base: fato Silver + dimensões de produto e categoria
# Inclui colunas auxiliares de trimestre e valores brutos para os KPIs
df_base = (
    df_silver_final.alias("f")
    .join(df_dim_produtos.alias("p"), on="sku", how="left")
    .join(df_categorias_hierarquia.alias("c"), on="id_categoria", how="left")
    .withColumn("trimestre", ceil(col("f.mes_particao") / 3))          # Q1=1, Q2=2, Q3=3, Q4=4
    .withColumn("valor_bruto_item", (col("f.preco_unitario") * col("f.quantidade")).cast("decimal(10,2)"))
    .withColumn("valor_desconto_total", (col("f.desconto_aplicado") * col("f.quantidade")).cast("decimal(10,2)"))
)

# ------------------------------------------------------------
# 2. CÁLCULO DOS KPIs
# ------------------------------------------------------------
timestamp_formatado = date_format(current_timestamp(), "yyyy-MM-dd HH:mm:ss")

# KPI 1: Receita líquida e taxa de desconto por categoria raiz (mensal)
df_kpi_categoria_mes = (
    df_base
    .groupBy("f.ano_particao", "f.mes_particao", "c.nome_categoria_raiz")
    .agg(
        round(sum("f.valor_liquido_item"), 2).alias("receita_liquida_total"),
        round(sum("valor_desconto_total") / sum("valor_bruto_item"), 4).alias("taxa_desconto_media")
    )
    .withColumn("gold_processed_at", timestamp_formatado)
)

# KPI 2: Top 10 SKUs por trimestre (receita e quantidade)
# CORREÇÃO (Negócio 7): a versão anterior trazia TODOS os SKUs do trimestre,
# sem nenhum corte de "top 10". Agora calculamos o ranking por receita e por
# quantidade separadamente (um SKU pode estar no top 10 de um e não do outro)
# e mantemos a união dos dois — cada linha mostra ambos os ranks, então o
# time comercial consegue filtrar por qualquer um dos dois critérios.
window_sku_receita    = Window.partitionBy("ano_particao", "trimestre").orderBy(col("receita_total").desc())
window_sku_quantidade = Window.partitionBy("ano_particao", "trimestre").orderBy(col("quantidade_total").desc())

df_kpi_sku_trimestre = (
    df_base
    .groupBy("f.ano_particao", "trimestre", "sku")
    .agg(
        round(sum("f.valor_liquido_item"), 2).alias("receita_total"),
        sum("f.quantidade").alias("quantidade_total")
    )
    .withColumn("rank_receita", row_number().over(window_sku_receita))
    .withColumn("rank_quantidade", row_number().over(window_sku_quantidade))
    .filter((col("rank_receita") <= 10) | (col("rank_quantidade") <= 10))
    .withColumn("gold_processed_at", timestamp_formatado)
)

# KPI 3: Complexidade do carrinho — média de itens por pedido (mensal)
# CORREÇÃO (Negócio 9): a versão anterior somava a coluna `quantidade`
# (unidades OU kg/L, conforme a unidade de medida do produto — Regra 2 da
# Silver), misturando grandezas diferentes e medindo "volume comprado" em
# vez de "complexidade do carrinho". Trocado para count(id_item_pedido):
# número de linhas/produtos distintos por pedido, que é o que de fato
# representa a complexidade do carrinho.
df_kpi_complexidade_carrinho = (
    df_silver_final
    .groupBy("ano_particao", "mes_particao", "id_pedido")
    .agg(count("id_item_pedido").alias("qtd_itens_pedido"))
    .groupBy("ano_particao", "mes_particao")
    .agg(round(avg("qtd_itens_pedido"), 2).alias("media_itens_por_pedido"))
    .withColumn("gold_processed_at", timestamp_formatado)
)

# KPI 4: Crescimento YoY por subcategoria via Window Function
# CORREÇÃO: particionamento trocado de nome_subcategoria para id_categoria
# (chave real da subcategoria). Duas subcategorias de raízes diferentes
# podem ter o mesmo nome (ex.: "Acessórios" em Eletrônicos e em Moda); usar
# o nome como chave de partição misturaria a receita das duas. nome_subcategoria
# continua na tabela apenas para exibição.
# lag() busca a receita do ano anterior para a mesma subcategoria
window_yoy = Window.partitionBy("id_categoria").orderBy("ano_particao")
df_kpi_yoy_subcategoria = (
    df_base
    .groupBy("f.ano_particao", "id_categoria", "c.nome_subcategoria")
    .agg(round(sum("f.valor_liquido_item"), 2).alias("receita_ano_atual"))
    .withColumn("receita_ano_anterior", lag("receita_ano_atual").over(window_yoy))
    .withColumn("crescimento_yoy_perc",
        round(((col("receita_ano_atual") - col("receita_ano_anterior")) / col("receita_ano_anterior")) * 100, 2)
    )
    .withColumn("gold_processed_at", timestamp_formatado)
)

# ------------------------------------------------------------
# 3. ADICIONA DIA DE EXECUÇÃO EM TODOS OS DATA MARTS
# Coluna usada como chave de controle no delete antes do append.
# Garante que reprocessamentos do mesmo dia não gerem duplicatas
# e que cargas de semanas anteriores do mesmo mês sejam preservadas.
# ------------------------------------------------------------
df_kpi_categoria_mes         = df_kpi_categoria_mes.withColumn("dia_exec", lit(DIA_EXEC))
df_kpi_sku_trimestre         = df_kpi_sku_trimestre.withColumn("dia_exec", lit(DIA_EXEC))
df_kpi_complexidade_carrinho = df_kpi_complexidade_carrinho.withColumn("dia_exec", lit(DIA_EXEC))
df_kpi_yoy_subcategoria      = df_kpi_yoy_subcategoria.withColumn("dia_exec", lit(DIA_EXEC))

# ------------------------------------------------------------
# 4. GRAVAÇÃO NO DELTA LAKE — DELETE + APPEND (IDEMPOTENTE)
# ------------------------------------------------------------
# CORREÇÃO (bug de idempotência): o loop genérico anterior usava sempre
# `WHERE ano_particao = ... AND mes_particao = ... AND dia_exec = ...`
# para TODAS as tabelas. gold_kpi_sku_trimestre não tem mes_particao (tem
# trimestre) e gold_kpi_yoy_subcategoria não tem nem mes_particao nem
# trimestre — o DELETE falhava com erro de coluna inexistente, o erro era
# engolido pelo try/except genérico, e a tabela nunca era limpa: cada
# execução semanal só acumulava mais uma cópia dos dados por cima.
#
# Agora cada tabela informa explicitamente quais colunas identificam o
# "lote" que deve ser substituído (delete_cols), e o DELETE é construído
# dinamicamente a partir delas — mesmo padrão já usado no pipeline de
# ecommerce_pedidos.
print("Gravando os Data Marts na camada Gold...")


def sql_literal(valor):
    if valor is None:
        return "NULL"
    if isinstance(valor, str):
        return "'" + valor.replace("'", "''") + "'"
    return str(valor)


def gravar_data_mart_delta(tabela_nome, df_mart, delete_cols):
    path_gold = f"abfss://squad3@{storage_account}.dfs.core.windows.net/gold/{tabela_nome}"

    # Tenta registrar a tabela Delta existente para o DELETE.
    # Se a tabela ainda não existir (primeira carga), o try/except ignora e vai direto ao append.
    try:
        (
            spark.read
            .format("delta")
            .options(**adls_options)
            .load(path_gold)
            .createOrReplaceTempView(f"tmp_{tabela_nome}")
        )

        particoes_reprocessadas = df_mart.select(*delete_cols).distinct().collect()

        for particao in particoes_reprocessadas:
            condicoes = [
                f"{coluna} = {sql_literal(particao[coluna])}"
                for coluna in delete_cols
            ]
            spark.sql(f"""
                DELETE FROM delta.`{path_gold}`
                WHERE {" AND ".join(condicoes)}
            """)

        print(f"Delete executado: {tabela_nome} ({len(particoes_reprocessadas)} particao(oes)).")
    except Exception:
        print(f"Primeira carga detectada — sem delete: {tabela_nome}")

    # Append limpo — sem duplicatas
    (
        df_mart.write
        .format("delta")
        .mode("append")
        .option("mergeSchema", "true")  # Permite evoluções de schema sem reprocessamento
        .options(**adls_options)
        .save(path_gold)
    )
    print(f"Data Mart atualizado no Data Lake: {tabela_nome}")


data_marts_config = {
    "gold_kpi_receita_desconto_categoria": {
        "df": df_kpi_categoria_mes,
        "delete_cols": ["ano_particao", "mes_particao", "dia_exec"],
    },
    "gold_kpi_sku_trimestre": {
        "df": df_kpi_sku_trimestre,
        "delete_cols": ["ano_particao", "trimestre", "dia_exec"],
    },
    "gold_kpi_complexidade_carrinho": {
        "df": df_kpi_complexidade_carrinho,
        "delete_cols": ["ano_particao", "mes_particao", "dia_exec"],
    },
    "gold_kpi_yoy_subcategoria": {
        "df": df_kpi_yoy_subcategoria,
        "delete_cols": ["ano_particao", "dia_exec"],
    },
}

for tabela_nome, config_mart in data_marts_config.items():
    gravar_data_mart_delta(
        tabela_nome=tabela_nome,
        df_mart=config_mart["df"],
        delete_cols=config_mart["delete_cols"],
    )

print("\nSUCESSO! Data Marts sumarizados e salvos na Gold com sucesso.")


## 5. 🗄️ Exportação para SQL Server

Etapa final do pipeline: os 4 Data Marts calculados na camada Gold são **exportados para o SQL Server** no schema `squad3`, tornando os KPIs disponíveis para consumo por ferramentas de BI e relatórios.

### Estratégia de carga

- **Modo:** `append` — novos registros são inseridos sem sobrescrever o histórico existente
- **Conector:** driver nativo `sqlserver` do Spark (sem JDBC URL explícita)
- **Tratamento de erros:** cada tabela é carregada de forma independente via `try/except`, garantindo que a falha em uma tabela não interrompa as demais

### Tabelas de destino no SQL Server

| Tabela SQL Server | Data Mart de origem |
|-------------------|---------------------|
| `squad3.gold_kpi_receita_desconto_categoria` | `df_kpi_categoria_mes` |
| `squad3.gold_kpi_sku_trimestre` | `df_kpi_sku_trimestre` |
| `squad3.gold_kpi_complexidade_carrinho` | `df_kpi_complexidade_carrinho` |
| `squad3.gold_kpi_yoy_subcategoria` | `df_kpi_yoy_subcategoria` |

> ⚠️ As credenciais do SQL Server (`SQL_HOST`, `SQL_DATABASE`, `SQL_USERNAME`, `SQL_PASSWORD`) são carregadas via `load_dotenv()` e nunca expostas no código.

In [0]:
# ============================================================
# 5. EXPORTAÇÃO DE DATA MARTS PARA O SQL SERVER (PRODUÇÃO)
# ============================================================

print("Iniciando a exportação dos Data Marts para o SQL Server (MODO APPEND)...")

# 1. Carrega as credenciais do ambiente
load_dotenv("../env")
jdbc_hostname = os.getenv("SQL_HOST")
jdbc_port = "1433"  # Porta padrão do SQL Server
jdbc_database = os.getenv("SQL_DATABASE")
jdbc_username = os.getenv("SQL_USERNAME")
jdbc_password = os.getenv("SQL_PASSWORD")

def carregar_no_sql_server(df, tabela_sql, mode="append"):
    """
    Grava um DataFrame PySpark em uma tabela do SQL Server via conector nativo.

    Parâmetros:
        df (DataFrame): DataFrame Spark a ser exportado.
        tabela_sql (str): Nome da tabela de destino no formato 'schema.tabela'.
        mode (str): Modo de escrita Spark ('append' por padrão).
                    Use 'overwrite' apenas em reprocessamentos completos controlados.
    """
    try:
        (
            df.write
            .format("sqlserver")
            .option("host", jdbc_hostname)
            .option("port", jdbc_port)
            .option("database", jdbc_database)
            .option("dbtable", tabela_sql)
            .option("user", jdbc_username)
            .option("password", jdbc_password)
            .mode(mode) 
            .save()
        )
        print(f"SUCESSO! Dados adicionados em {tabela_sql}")
    except Exception as e:
        print(f"Erro ao atualizar {tabela_sql}:\n{e}")

# 2. Mapeamento das tabelas SQL de destino para os DataFrames Gold
processamento = {
    "squad3.gold_kpi_receita_desconto_categoria": df_kpi_categoria_mes,
    "squad3.gold_kpi_sku_trimestre": df_kpi_sku_trimestre,
    "squad3.gold_kpi_complexidade_carrinho": df_kpi_complexidade_carrinho,
    "squad3.gold_kpi_yoy_subcategoria": df_kpi_yoy_subcategoria
}

# 3. Loop de carga — cada tabela é tratada independentemente para isolamento de erros
for tabela_destino, df_kpi in processamento.items():
    print(f"Iniciando carga: {tabela_destino}")
    carregar_no_sql_server(df_kpi, tabela_destino, mode="append")

print("\nTodas as tabelas foram atualizadas no schema squad3 do banco de dados!")

## 6. 📊 Data Quality — Sanidade Analítica (Camada Gold)

Esta etapa atua como uma barreira final de qualidade para os KPIs gerados. Ela garante que as regras de negócio e as agregações matemáticas aplicadas na camada Gold não produziram resultados analiticamente impossíveis antes de os dados serem exportados para o SQL Server.

### Regras de Validação Aplicadas

| Métrica Analítica | Regra de Negócio | Tabela Validada |
|-------------------|------------------|-----------------|
| **Receita Agrupada Negativa** | O total da receita líquida sumarizada por categoria e mês não pode ser menor que zero. | `gold_kpi_receita_desconto_categoria` |
| **Taxa de Desconto Inválida** | A taxa média de desconto deve obrigatoriamente estar entre 0% e 100% (0.0 a 1.0). | `gold_kpi_receita_desconto_categoria` |
| **Média de Itens/Pedido < 1** | O ticket de complexidade do carrinho deve apresentar, logicamente, no mínimo a média de 1 item por pedido. | `gold_kpi_complexidade_carrinho` |

> 💡 **Monitoramento Visual:** Os resultados da validação são consolidados e exibidos em um painel gráfico inteligente. O status utiliza cores semânticas (**Verde** para aprovação total / **Vermelho** para anomalias) para facilitar a observabilidade operacional do pipeline.

In [0]:
# ============================================================
# 6. DATA QUALITY E OBSERVABILIDADE (Métricas e Gráficos)
# ============================================================

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pyspark.sql.functions import col, sum as _sum, when, count

print("Iniciando validação de Data Quality no DataFrame Silver (df_silver_final)...")

# 1. Coleta de Métricas de DQ via PySpark
total_linhas = df_silver_final.count()

# Verifica Completude e Validade
dq_metrics = df_silver_final.select(
    # Completude
    _sum(when(col("sku").isNull(), 1).otherwise(0)).alias("nulos_sku"),
    _sum(when(col("quantidade").isNull(), 1).otherwise(0)).alias("nulos_quantidade"),
    _sum(when(col("valor_liquido_item").isNull(), 1).otherwise(0)).alias("nulos_valor"),
    
    # Validade
    _sum(when(col("quantidade") <= 0, 1).otherwise(0)).alias("qtd_invalida_negativa"),
    _sum(when(col("valor_liquido_item") < 0, 1).otherwise(0)).alias("valor_invalido_negativo")
).collect()[0]

# Verifica Unicidade (Contagem distinta vs Total)
total_distintos = df_silver_final.select("id_item_pedido").distinct().count()
duplicatas = total_linhas - total_distintos

# 2. Consolidação dos Resultados
dq_resultados = {
    "Nulos no SKU": dq_metrics["nulos_sku"],
    "Nulos na Quantidade": dq_metrics["nulos_quantidade"],
    "Nulos no Valor": dq_metrics["nulos_valor"],
    "Quantidades Inválidas (<=0)": dq_metrics["qtd_invalida_negativa"],
    "Valores Negativos (<0)": dq_metrics["valor_invalido_negativo"],
    "Chaves Duplicadas": duplicatas
}

# Converte para Pandas para facilitar a plotagem
pdf_dq = pd.DataFrame(list(dq_resultados.items()), columns=["Regra de Validação", "Total de Falhas"])

# Exibe o status geral
print(f"Total de registros avaliados: {total_linhas}")
display(pdf_dq)

# 3. Plotagem dos Gráficos (Matplotlib/Seaborn)
sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Gráfico 1: Barras com o total de falhas por regra
sns.barplot(
    data=pdf_dq, 
    x="Total de Falhas", 
    y="Regra de Validação", 
    palette="Reds_r", 
    ax=axes[0]
)
axes[0].set_title("Ocorrências de Falhas de Qualidade por Regra", fontsize=14, fontweight="bold")
axes[0].set_xlabel("Número de Registros Falhos")
axes[0].set_ylabel("")

# Gráfico 2: Saúde Geral dos Dados (Pizza)
total_falhas = pdf_dq["Total de Falhas"].sum()
registros_saudaveis = total_linhas - total_falhas

# Evita gráfico distorcido se as falhas forem maiores que o total de linhas (cenário irreal na Silver, mas previne bugs de plotagem)
registros_saudaveis = max(0, registros_saudaveis) 

labels = ['Registros Saudáveis', 'Registros com Falha']
sizes = [registros_saudaveis, total_falhas]
colors = ['#2ca02c', '#d62728']
explode = (0.1, 0) if total_falhas > 0 else (0, 0)

axes[1].pie(
    sizes, 
    explode=explode, 
    labels=labels, 
    colors=colors, 
    autopct='%1.2f%%', 
    shadow=True, 
    startangle=140,
    textprops={'fontsize': 12}
)
axes[1].set_title("Saúde Geral da Tabela (Silver)", fontsize=14, fontweight="bold")

plt.tight_layout()
plt.show()

# 4. Alerta Condicional (Opcional, mas recomendado)
if total_falhas > 0:
    print(f"⚠️ ATENÇÃO: Foram encontradas {total_falhas} anomalias nos dados. Verifique o gráfico acima.")
else:
    print("✅ SUCESSO: Nenhuma anomalia de Data Quality encontrada. Os dados estão consistentes.")

    

In [0]:
# ============================================================
# 6.5. DATA QUALITY - SANIDADE ANALÍTICA (Camada Gold)
# ============================================================

print("Validando sanidade dos KPIs na Camada Gold...")

# Exemplo de validação no KPI: Receita e Desconto por Categoria
dq_gold_categoria = df_kpi_categoria_mes.select(
    # A receita total agrupada não pode ser negativa
    _sum(when(col("receita_liquida_total") < 0, 1).otherwise(0)).alias("kpi_receita_negativa"),
    
    # A taxa de desconto média não deve ser maior que 100% (1.0) ou negativa
    _sum(when((col("taxa_desconto_media") < 0) | (col("taxa_desconto_media") > 1), 1).otherwise(0)).alias("kpi_taxa_invalida")
).collect()[0]

# Validação na tabela de Complexidade do Carrinho
dq_gold_carrinho = df_kpi_complexidade_carrinho.select(
    # A média de itens por pedido tem que ser no mínimo 1
    _sum(when(col("media_itens_por_pedido") < 1, 1).otherwise(0)).alias("media_itens_invalida")
).collect()[0]

# Consolidação dos resultados da Gold
gold_resultados = {
    "Receita Agrupada Negativa": dq_gold_categoria["kpi_receita_negativa"],
    "Taxa de Desconto > 100% ou < 0": dq_gold_categoria["kpi_taxa_invalida"],
    "Média de Itens/Pedido < 1": dq_gold_carrinho["media_itens_invalida"]
}

pdf_gold_dq = pd.DataFrame(list(gold_resultados.items()), columns=["Métrica Analítica (Gold)", "Total de Falhas"])
display(pdf_gold_dq)

total_falhas_gold = pdf_gold_dq["Total de Falhas"].sum()

if total_falhas_gold > 0:
    print(f"ATENÇÃO: Anomalias detectadas nos KPIs da Gold. Total de falhas: {total_falhas_gold}")
else:
    print("SUCESSO: Data Marts da camada Gold passaram nas verificações de sanidade.")

In [0]:
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import MaxNLocator

print("Gerando painel visual de qualidade da Camada Gold...")

# Configuração de estilo do Seaborn
sns.set_theme(style="whitegrid", rc={"axes.spines.right": False, "axes.spines.top": False})

# Criação da figura
fig, ax = plt.subplots(figsize=(10, 4))

# Lógica de cores: Verde para 0 falhas, Vermelho se houver > 0
cores = ["#2ca02c" if val == 0 else "#d62728" for val in pdf_gold_dq["Total de Falhas"]]

# Plotagem do gráfico de barras horizontais
sns.barplot(
    data=pdf_gold_dq,
    x="Total de Falhas",
    y="Métrica Analítica (Gold)",
    palette=cores,
    ax=ax
)

# Ajustes estéticos (Títulos e Eixos)
ax.set_title("Status da Validação de Sanidade (Camada Gold)", fontsize=14, fontweight="bold", pad=15)
ax.set_xlabel("Número de Ocorrências (Falhas)", fontsize=12, fontweight="bold")
ax.set_ylabel("")

# Garante que o eixo X mostre apenas números inteiros (não faz sentido "0.5 falhas")
ax.xaxis.set_major_locator(MaxNLocator(integer=True))

# Adiciona os rótulos de dados (os números) na ponta de cada barra
for i in ax.containers:
    ax.bar_label(i, padding=5, fontsize=11, fontweight="bold")

# Mostra o gráfico
plt.tight_layout()
plt.show()

In [0]:
# # ============================================================
# # TRUNCATE DA TABELA SILVER (VIA DATAFRAME API)
# # ============================================================
# print(f"Iniciando Truncate (Overwrite Vazio) da tabela Silver: {path_silver_itens}")

# try:
#     # 1. Cria um DataFrame completamente vazio, mas com a estrutura exata da sua tabela
#     df_vazio = spark.createDataFrame([], df_silver_final.schema)

#     # 2. Faz o Overwrite (que atua exatamente como um Truncate físico no Data Lake)
#     (
#         df_vazio.write
#         .format("delta")
#         .mode("overwrite")
#         .options(**adls_options) # Usa as credenciais que já sabemos que funcionam
#         .partitionBy("ano_particao", "mes_particao")
#         .save(path_silver_itens)
#     )
#     print("✅ SUCESSO: Tabela truncada e limpa fisicamente no Data Lake.")
    
# except Exception as e:
#     print(f"❌ Erro: {e}")